# llminfer OpenAI-Compatible Serving (Colab)

Run `llminfer` as an OpenAI-compatible HTTP API with continuous batching.

## 0) Enable GPU runtime
In Colab: **Runtime -> Change runtime type -> GPU**

In [ ]:
!nvidia-smi

## 1) Clone + install with serving extras

In [ ]:
REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'  # TODO
TARGET_DIR = '/content/llminfer'

import os, shutil
if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone {REPO_URL} {TARGET_DIR}
%cd /content/llminfer
!pip -q install -U pip
!pip -q install -e '.[serve]'
!pip -q install requests

## 2) Start server in a background thread

In [ ]:
import threading
import time

import uvicorn
from llminfer import EngineConfig, InferenceEngine
from llminfer.api import create_openai_app
from llminfer.serving import ContinuousBatchScheduler

cfg = EngineConfig(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    max_batch_size=16,
    batch_timeout_ms=20,
)
engine = InferenceEngine(cfg)
scheduler = ContinuousBatchScheduler(engine, max_batch_size=16, batch_timeout_ms=20, max_queue_size=1024)
app = create_openai_app(engine=engine, scheduler=scheduler, model_alias='local-llminfer')

def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(5)
print('Server started on http://127.0.0.1:8000')

## 3) Non-streaming chat completion

In [ ]:
import requests

payload = {
  'model': 'local-llminfer',
  'messages': [
      {'role': 'system', 'content': 'You are concise.'},
      {'role': 'user', 'content': 'Tell me about GPUs in 4 bullet points.'}
  ],
  'max_tokens': 160,
  'temperature': 0.2
}
resp = requests.post('http://127.0.0.1:8000/v1/chat/completions', json=payload, timeout=180)
resp.raise_for_status()
resp.json()

## 4) Streaming chat completion (SSE)

In [ ]:
stream_payload = {
  'model': 'local-llminfer',
  'messages': [{'role': 'user', 'content': 'Explain KV cache in under 80 words.'}],
  'max_tokens': 120,
  'temperature': 0.2,
  'stream': True
}

with requests.post('http://127.0.0.1:8000/v1/chat/completions', json=stream_payload, stream=True, timeout=180) as r:
    r.raise_for_status()
    for line in r.iter_lines(decode_unicode=True):
        if not line:
            continue
        if line.startswith('data: '):
            payload = line[6:]
            if payload == '[DONE]':
                print('\n[done]')
                break
            obj = __import__('json').loads(payload)
            delta = obj['choices'][0].get('delta', {}).get('content', '')
            if delta:
                print(delta, end='', flush=True)

## 5) Health and metrics

In [ ]:
print(requests.get('http://127.0.0.1:8000/healthz', timeout=20).json())
metrics_text = requests.get('http://127.0.0.1:8000/metrics', timeout=20).text
print('\n'.join(metrics_text.splitlines()[:20]))